# Valio Aimo – Shortage Risk Model from Purchases & Sales

This notebook:

1. Builds **risk features** from purchase orders (PO shortages).
2. Creates a **master training set** (currently using a simulated sales block based on PO data — you should replace it with real sales & deliveries).
3. Trains a **RandomForestClassifier** to predict shortage risk.
4. Outputs **shortage probability scores** that an AI agent can use to trigger proactive actions.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings

# --- Setup ---
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## Part 1 – Load Purchase Data & Engineer Risk Features

Here we:

- Load `valio_aimo_purchases_junction_2025.csv`
- Aggregate by `(order_number, po_row_number)` to sum multiple receipts
- Create a PO shortage flag: `po_shortage = ordered_qty > total_received_qty`
- Derive historical risk features:

  - `product_risk`
  - `customer_risk`
  - `unit_risk`

In [11]:
print("--- Part 1: Building Risk Features from Purchases ---")

try:
    # Load the purchase data ("supplier/quality data")
    purchases_df = pd.read_csv('valio_aimo_purchases_junction_2025.csv')
    print(f"Loaded purchases: {purchases_df.shape[0]:,} rows")
except FileNotFoundError:
    print("FATAL ERROR: valio_aimo_purchases_junction_2025.csv not found.")
    print("Please make sure it's uploaded and accessible.")
else:
    # 1. Aggregate by the unique PO line to sum multiple receipts
    po_agg = purchases_df.groupby(['order_number', 'po_row_number']).agg(
        customer_number=('customer_number', 'first'),
        product_code=('product_code', 'first'),
        po_created_date=('po_created_date', 'first'),
        ordered_qty=('ordered_qty', 'first'),
        total_received_qty=('received_qty', 'sum'),
        unit=('unit', 'first')
    ).reset_index()

    # 2. Create the supplier-side shortage flag
    po_agg['po_shortage'] = po_agg['ordered_qty'] > po_agg['total_received_qty']

    # 3. Calculate historical risk features (the "quality data")
    print("Calculating historical risk scores...")
    product_risk = po_agg.groupby('product_code')['po_shortage'].mean().rename('product_risk')
    customer_risk = po_agg.groupby('customer_number')['po_shortage'].mean().rename('customer_risk')
    unit_risk = po_agg.groupby('unit')['po_shortage'].mean().rename('unit_risk')

    print(f"Created {len(product_risk)} product risk scores.")
    print(f"Created {len(customer_risk)} customer risk scores.")

    print("\nTop 5 Riskiest Products (from PO data):")
    print(product_risk.sort_values(ascending=False).head())

--- Part 1: Building Risk Features from Purchases ---
Loaded purchases: 782,783 rows
Loaded purchases: 782,783 rows
Calculating historical risk scores...
Created 14110 product risk scores.
Created 621 customer risk scores.

Top 5 Riskiest Products (from PO data):
product_code
413026    1.0
411750    1.0
402465    1.0
402471    1.0
416634    1.0
Name: product_risk, dtype: float64
Calculating historical risk scores...
Created 14110 product risk scores.
Created 621 customer risk scores.

Top 5 Riskiest Products (from PO data):
product_code
413026    1.0
411750    1.0
402465    1.0
402471    1.0
416634    1.0
Name: product_risk, dtype: float64


## Part 2 – Create Master Training Dataset (Currently Uses Simulated Sales)

⚠️ **Important:**  
This section currently uses a **simulation** based on `po_agg` and PO shortages as the target.  
In your real pipeline, you should **replace this with true sales & deliveries data** and a proper shortage label.

Expected final columns for real data:

- `order_date`
- `customer_number`
- `product_code`
- `ordered_qty`
- `unit`
- `TARGET_is_shortage`

In [12]:
print("\n--- Part 2: Creating Master Training Set ---")

# Load the actual sales and deliveries data
try:
    sales_df = pd.read_csv('valio_aimo_sales_and_deliveries_junction_2025.csv')
    print(f"Loaded sales & deliveries: {sales_df.shape[0]:,} rows")
    print(f"Columns: {list(sales_df.columns)}")
except FileNotFoundError:
    print("FATAL ERROR: valio_aimo_sales_and_deliveries_junction_2025.csv not found.")
    print("Please make sure it's uploaded and accessible.")
else:
    # Display first few rows to inspect the data
    print("\nFirst few rows of sales data:")
    print(sales_df.head())


--- Part 2: Creating Master Training Set ---
Loaded sales & deliveries: 7,357,509 rows
Columns: ['order_number', 'order_created_date', 'order_created_time', 'requested_delivery_date', 'customer_number', 'order_row_number', 'product_code', 'order_qty', 'sales_unit', 'delivery_number', 'plant', 'storage_location', 'delivered_qty', 'transfer_number', 'warehouse_number', 'picking_confirmed_date', 'picking_confirmed_time', 'picking_picked_qty']

First few rows of sales data:
   order_number order_created_date  order_created_time  \
0      10000000         2024-09-01                 336   
1      10000000         2024-09-01                 336   
2      10000000         2024-09-01                 336   
3      10000000         2024-09-01                 336   
4      10000000         2024-09-01                 336   

  requested_delivery_date  customer_number  order_row_number  product_code  \
0              2024-09-02            33258                10        409510   
1              2024

## Part 3 – Merge Risk Features into Sales & Add Time Features

We now:

- Merge `product_risk`, `customer_risk`, `unit_risk` into the sales data
- Convert `order_date` to datetime
- Add:

  - `order_month`
  - `order_dayofweek`

In [13]:
print("Merging risk features into sales data...")

master_df = sales_df.merge(product_risk, on='product_code', how='left')
master_df = master_df.merge(customer_risk, on='customer_number', how='left')

# Only merge unit_risk if 'unit' column exists in sales_df
if 'unit' in sales_df.columns:
    master_df = master_df.merge(unit_risk, on='unit', how='left')

# Create shortage flag: delivered_qty < order_qty
master_df['TARGET_is_shortage'] = master_df['delivered_qty'] < master_df['order_qty']

# Time-based features from the sales order date
master_df['order_created_date'] = pd.to_datetime(master_df['order_created_date'])
master_df['order_month'] = master_df['order_created_date'].dt.month
master_df['order_dayofweek'] = master_df['order_created_date'].dt.dayofweek

# Handle missing values (e.g., a new product not in PO history)
master_df.fillna(0, inplace=True)

print("Master dataset created and ready for modeling.")
print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(f"Shortage cases: {master_df['TARGET_is_shortage'].sum()} out of {len(master_df)}")
master_df.head()

Merging risk features into sales data...
Master dataset created and ready for modeling.
Shape: (7357509, 23)
Columns: ['order_number', 'order_created_date', 'order_created_time', 'requested_delivery_date', 'customer_number', 'order_row_number', 'product_code', 'order_qty', 'sales_unit', 'delivery_number', 'plant', 'storage_location', 'delivered_qty', 'transfer_number', 'warehouse_number', 'picking_confirmed_date', 'picking_confirmed_time', 'picking_picked_qty', 'product_risk', 'customer_risk', 'TARGET_is_shortage', 'order_month', 'order_dayofweek']
Shortage cases: 164790 out of 7357509
Master dataset created and ready for modeling.
Shape: (7357509, 23)
Columns: ['order_number', 'order_created_date', 'order_created_time', 'requested_delivery_date', 'customer_number', 'order_row_number', 'product_code', 'order_qty', 'sales_unit', 'delivery_number', 'plant', 'storage_location', 'delivered_qty', 'transfer_number', 'warehouse_number', 'picking_confirmed_date', 'picking_confirmed_time', 'pic

,order_number,order_created_date,order_created_time,requested_delivery_date,customer_number,order_row_number,product_code,order_qty,sales_unit,delivery_number,...,transfer_number,warehouse_number,picking_confirmed_date,picking_confirmed_time,picking_picked_qty,product_risk,customer_risk,TARGET_is_shortage,order_month,order_dayofweek
0,10000000,2024-09-01,336,2024-09-02,33258,10,409510,5.0,ST,20000000.0,...,30000212.0,3001.0,2024-09-01,203837.0,5.0,0.24,0.0,False,9,6
1,10000000,2024-09-01,336,2024-09-02,33258,40,410914,12.0,ST,20000000.0,...,30000212.0,3001.0,2024-09-01,203734.0,12.0,0.00,0.0,False,9,6
2,10000000,2024-09-01,336,2024-09-02,33258,50,406587,4.0,ST,20000000.0,...,30000211.0,3001.0,2024-09-01,204149.0,4.0,0.00,0.0,False,9,6
3,10000000,2024-09-01,336,2024-09-02,33258,60,406588,4.0,ST,20000000.0,...,30000211.0,3001.0,2024-09-01,204124.0,4.0,0.00,0.0,False,9,6
4,10000000,2024-09-01,336,2024-09-02,33258,70,401369,8.0,BOT,20000000.0,...,30000211.0,3001.0,2024-09-01,205255.0,8.0,0.00,0.0,False,9,6


## Part 4 – Feature Selection, Encoding & Correlation Analysis

We:

1. Define **features** `X` and **target** `y`.
2. Encode categorical features (`product_code`, `customer_number`, `unit`) with `OrdinalEncoder`.
3. Compute & plot a **correlation matrix** to inspect relationships between features and the target.

In [14]:
print("\n--- Part 4: Feature Selection & Preparation ---")

# 1. Define Features (X) and Target (y)
y = master_df['TARGET_is_shortage']

# Build feature list dynamically based on available columns
feature_cols = [
    'product_code', 
    'customer_number', 
    'order_qty',  # Changed from ordered_qty to order_qty
    'order_month', 
    'order_dayofweek', 
    'product_risk',    # Engineered feature
    'customer_risk',   # Engineered feature
]

# Add sales_unit and unit_risk if available
if 'sales_unit' in master_df.columns:
    feature_cols.insert(2, 'sales_unit')
if 'unit_risk' in master_df.columns:
    feature_cols.append('unit_risk')

X = master_df[feature_cols]

# 2. Encode Categorical Features
categorical_cols = ['product_code', 'customer_number']
if 'sales_unit' in X.columns:
    categorical_cols.append('sales_unit')

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_encoded = X.copy()
X_encoded[categorical_cols] = encoder.fit_transform(X[categorical_cols])

# 3. Correlation Analysis
print("Generating correlation matrix (saved as 'correlation_matrix.png')...")
corr_df = X_encoded.copy()
corr_df['TARGET_is_shortage'] = y
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()
print("Saved 'correlation_matrix.png'")


--- Part 4: Feature Selection & Preparation ---
Generating correlation matrix (saved as 'correlation_matrix.png')...
Generating correlation matrix (saved as 'correlation_matrix.png')...
Saved 'correlation_matrix.png'
Saved 'correlation_matrix.png'


## Part 5 – Model Training and Validation

We:

1. Split the data into training and test sets using `train_test_split` with `stratify=y`.
2. Train a `RandomForestClassifier` with `class_weight='balanced'` (to address imbalance).
3. Evaluate performance using:

- `classification_report`
- `confusion_matrix` (saved as `confusion_matrix.png`)

In [15]:
print("\n--- Part 4: Model Training and Validation ---")

# 1. Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.3, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# 2. Train the Model
print("Training RandomForestClassifier...")
model = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model training complete.")

# 3. Validation
print("Evaluating model on test data...")
y_pred = model.predict(X_test)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues',
            xticklabels=['Predicted No Shortage', 'Predicted Shortage'],
            yticklabels=['Actual No Shortage', 'Actual Shortage'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.close()
print("Saved 'confusion_matrix.png'")


--- Part 4: Model Training and Validation ---
Training set size: 5150256
Test set size: 2207253
Training RandomForestClassifier...
Training set size: 5150256
Test set size: 2207253
Training RandomForestClassifier...
Model training complete.
Evaluating model on test data...
Model training complete.
Evaluating model on test data...

--- Classification Report ---

--- Classification Report ---
              precision    recall  f1-score   support

       False       0.98      0.99      0.99   2157816
        True       0.34      0.25      0.29     49437

    accuracy                           0.97   2207253
   macro avg       0.66      0.62      0.64   2207253
weighted avg       0.97      0.97      0.97   2207253

Saved 'confusion_matrix.png'
              precision    recall  f1-score   support

       False       0.98      0.99      0.99   2157816
        True       0.34      0.25      0.29     49437

    accuracy                           0.97   2207253
   macro avg       0.66      0.

## Part 6 – Risk Scores for AI Agent / Buffer Logic

We use `predict_proba()` to:

- Compute `predicted_shortage_risk` = probability of shortage for each test row.
- Show the **highest-risk true shortages**.
- Illustrate how an AI agent would use the risk score (threshold-based trigger).

In [16]:
print("\n--- Part 5: Generating 'Buffer Prediction' (Risk Scores) ---")

# Shortage probabilities (class 1)
probabilities = model.predict_proba(X_test)[:, 1]

results_df = X_test.copy()
results_df['actual_shortage'] = y_test.values
results_df['predicted_shortage_risk'] = probabilities

print("Example of risk scores for AI Agent (top 5 highest-risk true shortages):")
print(results_df[results_df['actual_shortage'] == True].sort_values(
    by='predicted_shortage_risk', ascending=False
).head(5))

print("\n--- How your AI agent would use this ---")
print("new_risk_score = model.predict_proba(new_order_features)[:, 1][0]")
print("if new_risk_score > 0.5:  # 0.5 is your threshold")
print("    trigger_ai_call(customer_number, product_code)")


--- Part 5: Generating 'Buffer Prediction' (Risk Scores) ---
Example of risk scores for AI Agent (top 5 highest-risk true shortages):
         product_code  customer_number  sales_unit  order_qty  order_month  \
1566726         304.0           3831.0         5.0        1.0           11   
814255         3669.0           4479.0         5.0        9.0           10   
4846980        8421.0           4561.0         5.0       16.0            5   
1510343        7263.0           1396.0        15.0        6.0           11   
1101198        7901.0           2925.0         5.0       26.4           10   

         order_dayofweek  product_risk  customer_risk  actual_shortage  \
1566726                0      0.068493            0.0             True   
814255                 1      0.344538            0.0             True   
4846980                6      0.280488            0.0             True   
1510343                3      0.066194            0.0             True   
1101198                3  